In [1]:
import json 
import numpy as np 
from pathlib import Path 

In [2]:
# Check Jenelle's format to see structure

path = '/mindhive/mcdermott/www/mturk_stimuli/audio_word_rec_metamers/' \
        'FOR_WEB_audio_experiment_6_load_experiment_info.json'

with open(path, 'r') as file:
    config = json.load(file)

In [3]:
len(config['random_selection_spectemp_model'])

200

In [4]:
len([ix for val in config['examples_per_model_and_layer'].values() for ix in val])

1400

In [5]:
config.keys()

dict_keys(['json_metadata', 'experiment_conditions', 'examples_per_model_and_layer', 'experiment_info', 'total_num_unique_stimuli', 'orig_condition', 'orig_replacement_condition', 'random_selection_spectemp_model'])

In [6]:
config['examples_per_model_and_layer'].keys()

dict_keys(['0_orig/0_orig', '0_orig/1_orig_replacement_copies', '1_spectemp_filters', '1_spectemp_filters/0_input_after_preproc', '1_spectemp_filters/1_filtered_signal', '1_spectemp_filters/2_spectempfilter_power', '1_spectemp_filters/3_avgpool', '1_spectemp_filters/4_final'])

### Specifying files in each condition:

In the metamers experiments, all unique files are present in each condition. For example:

In [7]:
all_tokens = np.vstack([vals for vals in config['examples_per_model_and_layer'].values()
                        if not isinstance(vals, dict)])

(all_tokens == all_tokens[0]).all()

True

We will do the same in making our config for the speech in noise task.

___
# Make config 

In [8]:
## Handmake Json for experiment 

num_catch_trials = 12

# get condition list from dir
parent_dir = Path('/mindhive/mcdermott/www/mturk_stimuli/imgriff/timit_attentive_listening_task/')
experiment_conditions = [cond_path.stem for cond_path in parent_dir.glob('*snr') if '-15' not in cond_path.stem
                        ]

timit_config = {}
timit_config['json_metadata'] = 'Pilot TIMIT attentional listening task: constructed 10/19/2022.'

timit_config['experiment_conditions'] = experiment_conditions
timit_config['experiment_info'] = {'task': ['timit_attentive_listening'],
         'experiment_folder': parent_dir.as_posix(),
          'num_catch_trials': num_catch_trials,
          'dataset': 'attn_task_dataset_00'}

timit_config['total_num_unique_stimuli'] = 400
timit_config


{'json_metadata': 'Pilot TIMIT attentional listening task: constructed 10/19/2022.',
 'experiment_conditions': ['two_distractor_3dB_snr',
  'four_distractor_3dB_snr',
  'one_distractor_-6dB_snr',
  'two_distractor_-6dB_snr',
  'one_distractor_3dB_snr',
  'four_distractor_-6dB_snr',
  'one_distractor_0dB_snr',
  'two_distractor_0dB_snr',
  'four_distractor_0dB_snr',
  'ssn_distractor_-6dB_snr',
  'two_distractor_-3dB_snr',
  'one_distractor_-3dB_snr',
  'ssn_distractor_0dB_snr',
  'ssn_distractor_3dB_snr',
  'ssn_distractor_-3dB_snr',
  'four_distractor_-3dB_snr'],
 'experiment_info': {'task': ['timit_attentive_listening'],
  'experiment_folder': '/mindhive/mcdermott/www/mturk_stimuli/imgriff/timit_attentive_listening_task',
  'num_catch_trials': 12,
  'dataset': 'attn_task_dataset_00'},
 'total_num_unique_stimuli': 400}

##### No randomization here, just load pre-shuffled and generated stimuli

In [9]:
exp_files = []
for condition in experiment_conditions:
    cond_dir = parent_dir / condition
    files = [f"{condition}/{cond_path.stem}.wav" for cond_path in cond_dir.glob('set_00_*.wav')]
    exp_files.extend(files)

In [10]:
timit_config['experiment_files'] = exp_files

In [11]:
## Catch trials
catch_dir = parent_dir / 'catch_trials'
catch_trials = [f"catch_trials/{file.stem}.wav" for file in catch_dir.glob('set_00_*.wav')]

In [12]:
catch_trials

['catch_trials/set_00_stim_402_word_27_speaker_m.wav',
 'catch_trials/set_00_stim_411_word_405_speaker_m.wav',
 'catch_trials/set_00_stim_408_word_494_speaker_m.wav',
 'catch_trials/set_00_stim_407_word_778_speaker_m.wav',
 'catch_trials/set_00_stim_403_word_71_speaker_m.wav',
 'catch_trials/set_00_stim_410_word_257_speaker_m.wav',
 'catch_trials/set_00_stim_400_word_730_speaker_f.wav',
 'catch_trials/set_00_stim_401_word_764_speaker_m.wav',
 'catch_trials/set_00_stim_406_word_313_speaker_m.wav',
 'catch_trials/set_00_stim_405_word_169_speaker_m.wav',
 'catch_trials/set_00_stim_409_word_690_speaker_m.wav',
 'catch_trials/set_00_stim_404_word_668_speaker_m.wav']

In [13]:
timit_config['catch_trials'] = catch_trials

In [14]:
timit_config['catch_trials']

['catch_trials/set_00_stim_402_word_27_speaker_m.wav',
 'catch_trials/set_00_stim_411_word_405_speaker_m.wav',
 'catch_trials/set_00_stim_408_word_494_speaker_m.wav',
 'catch_trials/set_00_stim_407_word_778_speaker_m.wav',
 'catch_trials/set_00_stim_403_word_71_speaker_m.wav',
 'catch_trials/set_00_stim_410_word_257_speaker_m.wav',
 'catch_trials/set_00_stim_400_word_730_speaker_f.wav',
 'catch_trials/set_00_stim_401_word_764_speaker_m.wav',
 'catch_trials/set_00_stim_406_word_313_speaker_m.wav',
 'catch_trials/set_00_stim_405_word_169_speaker_m.wav',
 'catch_trials/set_00_stim_409_word_690_speaker_m.wav',
 'catch_trials/set_00_stim_404_word_668_speaker_m.wav']

In [15]:
out_file = parent_dir / "PILOT_attentive_listening_experiment_info_dataset_00.json"

with open(out_file, 'w') as f:
    json.dump(timit_config, f)

In [16]:
# out_file = '/mindhive/mcdermott/www/mturk_stimuli/imgriff/single_word_rec/pilot/PILOT_speech_in_noise_load_experiment_info.json'

# with open(out_file, 'w') as f:
#     json.dump(giga_config, f)